# Lab Assignment 3 — Task 2

## Converting a CFG to Chomsky Normal Form (CNF)

Many parsing algorithms, including CKY, require the grammar to be in **Chomsky Normal Form (CNF)**.

A grammar is in CNF if its production rules have one of these forms:

1. `A -> B C` — a non-terminal produces two non-terminals.
2. `A -> 'word'` — a non-terminal produces a single terminal.

## PART 1 : The Challenge of Ambiguity & CNF Constraints

Consider:

> **I saw the man with a telescope.**

This sentence has two possible interpretations:

1. I saw a man who was holding a telescope.
2. I used a telescope to see the man.

A PCFG assigns probabilities to grammar rules. The probability of a complete parse tree is obtained by multiplying the probabilities of all production rules used in that tree.

### Your task

1. **Convert the given PCFG into strict CNF.** Identify the rule that violates CNF and replace it with equivalent binary rule(s). Preserve the probability of the original alternative.
2. **Parse the sentence using the CNF PCFG and `ViterbiParser`.** Print the most probable parse tree and its total probability.
3. **Explain mathematically why the parser selected this interpretation over the alternative.** Show the rule probabilities used in the selected tree and compare the resulting probability with the competing interpretation.

In [ ]:
pcfg_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP PP [0.3]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

sentence = "I saw the man with a telescope"

In [ ]:
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser

# Convert the non-CNF rule:
#     VP -> V NP PP [0.3]
# into binary rules. Choose an intermediate non-terminal and
# distribute the probability so that the original derivation
# keeps the same probability.

# Example:
#     VP -> V VP_PP [0.3]
#     VP_PP -> NP PP [1.0]

cnf_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

In [ ]:
# Parse CNF PCFG grammar and run ViterbiParser
pcfg = PCFG.fromstring(cnf_grammar_str)
viterbi = ViterbiParser(pcfg)
tokens = sentence.split()

parses = list(viterbi.parse(tokens))
print('Number of parse trees found:', len(parses))

if parses:
    most_probable_tree = parses[0]
    print('Total Probability (P(Tree)):', most_probable_tree.prob())
    print('')
    print('Most Probable Parse Tree:')
    print(most_probable_tree)
    print('')
    print('Visual Representation:')
    most_probable_tree.pretty_print()


### Analysis requirement

Do not hard-code the final probability.

Use the parse tree returned by the parser to determine which rules were selected. Then calculate the probability of the competing interpretation from the corresponding rule probabilities.

Your explanation should make clear that **ViterbiParser selects the parse with the highest product of rule probabilities**.

In [ ]:
# Mathematical Comparison of Interpretations
# 1. Interpretation 1 (NP Attachment: "the man with a telescope")
# Derivation:
# S -> NP VP [1.0]
#   NP -> "I" [0.1]
#   VP -> V NP [0.7]
#     V -> "saw" [1.0]
#     NP -> NP PP [0.6]
#       NP -> Det N [0.3] (Det -> "the" [0.8], N -> "man" [0.5])
#       PP -> P NP [1.0]  (P -> "with" [1.0], NP -> Det N [0.3], Det -> "a" [0.2], N -> "telescope" [0.5])

p_np_man = 0.3 * 0.8 * 0.5         # 0.12
p_pp = 1.0 * 1.0 * (0.3 * 0.2 * 0.5) # 0.03
p_np_telescope_phrase = 0.6 * p_np_man * p_pp # 0.6 * 0.12 * 0.03 = 0.00216

prob_interp1 = 1.0 * 0.1 * (0.7 * 1.0 * p_np_telescope_phrase)

# 2. Interpretation 2 (VP Attachment: "saw ... with a telescope")
# Derivation in CNF:
# S -> NP VP [1.0]
#   NP -> "I" [0.1]
#   VP -> V VP_PP [0.3]
#     V -> "saw" [1.0]
#     VP_PP -> NP PP [1.0]
#       NP -> Det N [0.3] (Det -> "the" [0.8], N -> "man" [0.5])
#       PP -> P NP [1.0]  (P -> "with" [1.0], NP -> Det N [0.3], Det -> "a" [0.2], N -> "telescope" [0.5])

prob_interp2 = 1.0 * 0.1 * (0.3 * 1.0 * 1.0 * p_np_man * p_pp)

print("-" * 60)
print("Interpretation 1 (NP attachment): P =", prob_interp1)
print("Interpretation 2 (VP attachment): P =", prob_interp2)
print("-" * 60)
print("Why ViterbiParser chose Interpretation 1:")
print("1. Both trees share identical probability products for:")
print("   S -> NP VP (1.0), NP -> I (0.1), V -> saw (1.0),")
print("   NP (the man) = 0.12, and PP (with a telescope) = 0.03.")
print("2. The difference is solely the attachment structure:")
print("   - Interp 1 (NP attach): P(VP->V NP) * P(NP->NP PP) = 0.7 * 0.6 = 0.42")
print("   - Interp 2 (VP attach): P(VP->V VP_PP) * P(VP_PP->NP PP) = 0.3 * 1.0 = 0.30")
print("3. Since 0.42 > 0.30, P(Interp 1) = 0.0001512 > P(Interp 2) = 0.0001080.")
print("ViterbiParser selects the tree that maximizes the joint rule probabilities.")


# Part 2: Advanced Dependency Parsing Using spaCy

Use the **spaCy NLP library** to perform dependency parsing and graph traversal on the following sentence:

> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**


## 1. Initialize and Parse

Load the `en_core_web_sm` model in spaCy and process the sentence.

Return the parsed object and store it as **`parsed_doc`**.

In [ ]:
import spacy
from collections import deque
from spacy import displacy

# Load spaCy small English model
nlp = spacy.load("en_core_web_sm")

sentence_dep = "The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation."
parsed_doc = nlp(sentence_dep)

print("Sentence:", parsed_doc.text)


## 2. Isolate the Root

Create a function that accepts **`parsed_doc` as its only parameter**.

Iterate through the document to find the structural center of the sentence, where the dependency tag is `"ROOT"`.

Return this token and save it as **`root_node`**.


In [ ]:
def isolate_root(parsed_doc):
    """
    Finds and returns the root token where dep_ == "ROOT".
    Accepts parsed_doc as its only parameter.
    """
    for token in parsed_doc:
        if token.dep_ == "ROOT":
            return token
    return None

root_node = isolate_root(parsed_doc)
print("Root Node:", root_node.text, f"(POS: {root_node.pos_}, Dep: {root_node.dep_})")


## 3. Extract the Graph Terminals

Write a function that requires **both `root_node` and `parsed_doc`** as inputs.

1. Check the `.children` of `root_node` to find the nominal subject (`nsubj` dependency), isolating **`researchers`** as `start_node`.
2. Find the token whose text is **`mutation`** in `parsed_doc`, isolating it as `end_node`.
3. Return both nodes.

In [ ]:
def extract_terminals(root_node, parsed_doc):
    """
    Finds start_node (nsubj of root_node) and end_node (token "mutation").
    Requires both root_node and parsed_doc as inputs.
    """
    start_node = None
    end_node = None
    
    # 1. Check children of root_node for nominal subject (nsubj)
    for child in root_node.children:
        if child.dep_ == "nsubj":
            start_node = child
            break
            
    # 2. Find token whose text is "mutation"
    for token in parsed_doc:
        if token.text.lower() == "mutation":
            end_node = token
            break
            
    return start_node, end_node

start_node, end_node = extract_terminals(root_node, parsed_doc)
print("Start Node:", start_node.text, f"(POS: {start_node.pos_}, Dep: {start_node.dep_})")
print("End Node  :", end_node.text, f"(POS: {end_node.pos_}, Dep: {end_node.dep_})")


## 4. Graph Traversal — Shortest Path

Treat the dependency parse as an **undirected graph**. From any token, you may move to:

- its `.head`, or
- any of its `.children`.

Write a **Breadth-First Search (BFS)** function that takes `start_node` and `end_node` as inputs.

Find the shortest path between the two words and return it as a list of tokens named **`dependency_path`**.

In [ ]:
def bfs_shortest_path(start_node, end_node):
    """
    Treats dependency tree as an undirected graph (traversing head and children).
    Finds and returns the shortest path between start_node and end_node.
    """
    if start_node is None or end_node is None:
        return []
        
    queue = deque([[start_node]])
    visited = {start_node}
    
    while queue:
        path = queue.popleft()
        current = path[-1]
        
        if current == end_node:
            return path
            
        # Neighbors in undirected graph: head and children
        neighbors = []
        if current.head and current.head != current:
            neighbors.append(current.head)
        neighbors.extend(list(current.children))
        
        for neighbor in neighbors:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])
                
    return []

dependency_path = bfs_shortest_path(start_node, end_node)
print("Shortest Dependency Path:", " -> ".join([t.text for t in dependency_path]))


## 5. Path Analysis

Write a function that accepts **`dependency_path`** as its input.

Iterate through **this list of tokens only**, not the whole sentence, and print their syntactic details in the following format:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| ... | ... | ... | ... |

This should extract only the most important syntactic skeleton connecting the two concepts.

In [ ]:
def analyze_path(dependency_path):
    """
    Prints syntactic details of tokens in dependency_path only.
    """
    print("=" * 60)
    print("Word            | POS      | Head            | Dependency")
    print("-" * 60)
    for token in dependency_path:
        print(f"{token.text:<15} | {token.pos_:<8} | {token.head.text:<15} | {token.dep_:<12}")
    print("=" * 60)

analyze_path(dependency_path)


## 6. Visual Verification

Pass the original **`parsed_doc`** into spaCy's `displacy` module to render the full dependency tree.

Use the resulting diagram to visually verify that the table generated in Step 5 matches the shortest path in the dependency tree.

In [ ]:
# Visual Verification using spaCy displaCy
displacy.render(parsed_doc, style="dep", jupyter=True, options={"distance": 110})
